# Capstone Assignment 2: Approach Deliverable

## Analyzing a Neo4j Graph Database via Cypher + GDS

**Assignment:** FAERS (FDA Adverse Event Reporting System) Graph Database Analysis

---

## 1. Introduction and Context

This assignment involves analyzing a healthcare graph database built from the FDA's Adverse Event Reporting System (FAERS) data. The FAERS database tracks adverse events and medication errors reported for approved drugs and therapeutic biologics, serving as a critical tool for post-market safety surveillance.

The graph contains eight node types:
- **Drug**: Medications that potentially contributed to adverse events
- **Case**: Patient demographic information
- **Reaction**: Specific adverse reactions experienced by patients
- **ReportSource**: Origin of the adverse event report
- **Outcome**: Long-term outcomes (e.g., Death, Hospitalization, Disability)
- **Therapy**: Description of prescribed therapy
- **Manufacturer**: Pharmaceutical company names
- **AgeGroup**: Patient age demographics

The analysis will progress through three phases:
1. **Exploratory Data Analysis (EDA)** using Cypher queries
2. **Deeper Analytical Questions** using advanced Cypher
3. **Graph Data Science (GDS)** analysis using Neo4j's GDS library

## Neo4j connection


In [2]:
import os
import pandas as pd
from neo4j import GraphDatabase

# Set env
URI = os.environ.get("NEO4J_URI")
USER = os.environ.get("NEO4J_USER")
PASSWORD = os.environ.get("NEO4J_PASSWORD")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

def run_query(query, parameters=None):
    """Run Cypher and return a pandas DataFrame. Use for MATCH, CALL gds.*, etc."""
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return pd.DataFrame([dict(record) for record in result])

# Show full content in DataFrames 
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

# hide warnings
import warnings
warnings.filterwarnings('ignore')


## 2. Overall Methodology

### 2.1 Iterative Discovery Approach

Given that the complete graph schema is not fully documented, I will adopt an **iterative discovery methodology**:

1. **Start Broad, Then Narrow**: Begin with high-level queries to understand graph structure, then drill down into specific relationships and properties
2. **Validate Assumptions**: Each query will inform the next, allowing me to validate my understanding of node properties and relationship patterns
3. **Document as I Go**: Maintain clear documentation of discovered schema elements, property names, and relationship types to inform subsequent analyses

### 2.2 Query Development Strategy

- **Incremental Complexity**: Start with simple `MATCH` and `RETURN` statements, then progressively add `WHERE` clauses, aggregations, and ordering
- **Performance Awareness**: For large result sets, use `LIMIT` clauses during exploration, then remove them for final results
- **Result Validation**: Cross-reference query results with domain knowledge about healthcare data to ensure logical consistency

---
## EDA: Graph Database Queries

The following exploratory queries establish the structure and content of the Healthcare (FAERS) graph.

### 1) Total number of nodes

**Query logic:** I match every node in the graph with `MATCH (n)` and return the count. This gives the total number of entities (nodes) in the Healthcare graph.

**Explanation:** Knowing the total node count establishes the scale of the database and provides a baseline for understanding how many drugs, cases, reactions, outcomes, and other entity types are represented overall.

**Interpretation of results:** The graph contains **11,381 nodes**, indicating a moderate-sized FAERS dataset suitable for exploratory and GDS analysis. This scale supports both Cypher-based queries and in-memory graph projections.

In [27]:
df = run_query("""
MATCH (n)
RETURN count(n) AS total_nodes;
""")
display(df)

,total_nodes
0,11381


### 2) All distinct graph labels

**Query logic:** The procedure `db.labels()` returns every node label (type) present in the database.

**Explanation:** Labels define the kinds of entities in the graph (e.g., Drug, Case, Reaction). Listing them confirms we have the expected eight FAERS entity types and reveals the exact label names to use in later queries.

**Interpretation of results:** The graph has **eight distinct node types** (AgeGroup, Case, Drug, Manufacturer, Outcome, Reaction, ReportSource, Therapy), matching the expected FAERS schema. These labels are used throughout the analytical and GDS sections.

In [28]:
df = run_query("""
CALL db.labels() YIELD label
RETURN label
ORDER BY label;
""")
display(df)

,label
0,AgeGroup
1,Case
2,Drug
3,Manufacturer
4,Outcome
5,Reaction
6,ReportSource
7,Therapy


### 3) Total number of relationships

**Query logic:** I match any pattern of the form source node to relationship to target node with `MATCH ()-[r]->()` and return the count of relationships. Each directed edge is counted once.

**Explanation:** The relationship count shows how densely the graph is connected. Together with the node count, it helps assess data volume and the ratio of links to entities, which is useful for understanding connectivity and planning analytical queries.

**Interpretation of results:** With **61,453 relationships**, the graph has roughly 5.4 edges per node on average, indicating moderate connectivity that supports path-based and GDS analyses.

In [29]:
df = run_query("""
MATCH ()-[r]->()
RETURN count(r) AS total_relationships;
""")
display(df)

,total_relationships
0,61453


### 4) All distinct relationship types

**Query logic:** I traverse all relationships with `MATCH ()-[r]->()`, use `DISTINCT type(r)` to get unique relationship type names, and order them alphabetically.

**Explanation:** Relationship types define how nodes are linked (e.g., Case-to-Reaction, Drug-to-Case). Listing them is necessary to write correct patterns in later Cypher queries and to understand how cases, drugs, reactions, outcomes, and other entities are connected in the FAERS model.

**Interpretation of results:** The **11 relationship types** include HAS_REACTION (Case→Reaction), IS_PRIMARY_SUSPECT, IS_SECONDARY_SUSPECT, IS_CONCOMITANT, IS_INTERACTING (Case→Drug), RESULTED_IN (Case→Outcome), and REGISTERED (Manufacturer→Case). These are central to the analytical and GDS queries.

In [30]:
df = run_query("""
MATCH ()-[r]->()
RETURN DISTINCT type(r) AS relationship_type
ORDER BY relationship_type;
""")
display(df)

,relationship_type
0,FALLS_UNDER
1,HAS_REACTION
2,IS_CONCOMITANT
3,IS_INTERACTING
4,IS_PRIMARY_SUSPECT
5,IS_SECONDARY_SUSPECT
6,PRESCRIBED
7,RECEIVED
8,REGISTERED
9,REPORTED_BY


### 5) Unique properties for each node type (label)

**Query logic:** The procedure `db.schema.nodeTypeProperties()` returns each node label together with the property names defined on that label. I aggregate by label with `collect(DISTINCT propertyName)` so that all properties for a given node type appear in one row.

**Explanation:** Each label can have different properties (e.g., Case might have `gender`, `age`; Reaction might have `description`). Knowing the exact property names per label is required to write correct filters and return values in the analytical and GDS sections that follow.

**Interpretation of results:** Case nodes include **primaryid**, **age**, **gender**, **reporterOccupation**, and **reportDate**, needed for demographic and community-based analysis. Reaction uses **description** for adverse-event text; Outcome uses **outcome**; Drug uses **name**. These property names are used in all subsequent queries.

In [63]:
df=run_query("""
CALL db.schema.nodeTypeProperties()
YIELD nodeLabels, propertyName
RETURN nodeLabels, collect(DISTINCT propertyName) AS properties
ORDER BY nodeLabels;
""")
display(df)

Received notification from DBMS server: <GqlStatusObject gql_status='01N62', status_description='warn: procedure or function execution warning. Execution of the procedure db.schema.nodeTypeProperties() generated the warning The field `propertyTypes` will change output format in the next major version.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='GENERIC', classification=<NotificationClassification.GENERIC: 'GENERIC'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'GENERIC', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL db.schema.nodeTypeProperties()\nYIELD nodeLabels, propertyName\nRETURN nodeLabels, collect(DISTINCT propertyName) AS properties\nORDER BY nodeLabels;\n'


,nodeLabels,properties
0,[AgeGroup],[ageGroup]
1,[Case],"[primaryid, age, ageUnit, gender, eventDate, reportDate, reporterOccupation]"
2,[Drug],"[name, primarySubstabce]"
3,[Manufacturer],[manufacturerName]
4,[Outcome],"[code, outcome]"
5,[Reaction],[description]
6,[ReportSource],"[name, code]"
7,[Therapy],[primaryid]


### 6) Distinct genders in Case node 'gender' property

**Query logic:** I match nodes with the `Case` label, filter to those where `gender` is not null, and return distinct values of `c.gender` ordered alphabetically.

**Explanation:** Case nodes represent patients; the `gender` property is a demographic used in later analysis (e.g., community detection or risk by subgroup). Listing distinct values shows the coding scheme (e.g., Male/Female, M/F) and reveals any data-quality issues or unexpected values.

**Interpretation of results:** The three gender values (**F**, **M**, **U**) use standard FAERS coding. Gender is populated and ready for demographic subgroup analysis and community detection.

In [32]:
df = run_query("""
MATCH (c:Case)
WHERE c.gender IS NOT NULL
RETURN DISTINCT c.gender AS gender
ORDER BY gender;
""")
display(df)

,gender
0,F
1,M
2,U


### 7) Number of distinct reactions

**Query logic:** I match all `Reaction` nodes, restrict to those with a non-null `description` (the property that holds the reaction text in this graph), and return the count of distinct `description` values.

**Explanation:** Reactions are the adverse events (e.g., "Pain", "Nausea") reported for cases. Counting distinct reaction descriptions gives the size of the "vocabulary" of adverse events in the data and supports later analyses such as most frequent reactions or patient similarity based on reaction profiles.

**Interpretation of results:** The **2,701 distinct reaction descriptions** indicate a rich adverse-event vocabulary, supporting frequency analysis and patient-similarity comparisons in the GDS section.

In [33]:
df = run_query("""
MATCH (r:Reaction)
WHERE r.description IS NOT NULL
RETURN count(DISTINCT r.description) AS distinct_reactions;
""")
display(df)

,distinct_reactions
0,2701


### 8) All distinct Outcome description values

**Query logic:** I match all `Outcome` nodes, filter to those with a non-null `outcome` property, and return distinct values of `o.outcome` in alphabetical order.

**Explanation:** Outcomes describe the long-term result of the adverse event (e.g., "Death", "Hospitalization", "Disability"). Listing distinct outcome values shows the exact strings used in the database, which is needed to correctly filter for the four most severe outcomes (Death, Life-Threatening, Disability, Hospitalization) in the deeper analytical questions.

**Interpretation of results:** The six outcome types include the four severe outcomes (**Death**, **Life-Threatening**, **Disability**, **Hospitalization - Initial or Prolonged**) required for the drug-severity analysis. The exact string for hospitalization must be used in the analytical queries.

In [34]:
df = run_query("""
MATCH (o:Outcome)
WHERE o.outcome IS NOT NULL
RETURN DISTINCT o.outcome AS outcome_description
ORDER BY outcome_description;
""")
display(df)

,outcome_description
0,Congenital Anomaly
1,Death
2,Disability
3,Hospitalization - Initial or Prolonged
4,Life-Threatening
5,Other Serious (Important Medical Event)


---
## Deeper Analytical Questions

The following queries use the graph structure discovered in EDA: node labels; relationship types (`HAS_REACTION`, `RESULTED_IN`, `REGISTERED`, and suspect types from Case to Drug); and property names (`Reaction.description`, `Outcome.outcome`, `Drug.name`, `Manufacturer.manufacturerName`). Note that Manufacturer links to Case via `REGISTERED`, not to Drug.

### 1) Top 20 most frequently occurring reactions

**Query logic:** I use the EDA-discovered pattern: cases link to reactions with `(c:Case)-[:HAS_REACTION]->(r:Reaction)`, and the reaction text is in `r.description`. I match all such case–reaction pairs, restrict to non-null `description`, and count how many times each reaction appears (one count per pair). I return `r.description` as `reaction` and `count(*)` as `frequency`, order by frequency descending, and take the top 20. Thus "frequency" is the number of (Case, Reaction) pairs, so reactions reported in more cases rank higher.

**Explanation:** This ranks adverse events by how often they are reported in the database. The result supports post-market surveillance (which reactions to monitor), labeling discussions, and comparison with other data sources. The top reactions reflect both true prevalence and reporting patterns.

**Interpretation of results:** **Fatigue** (303), **Product dose omission issue** (285), **Headache** (272), **Nausea** (256), and **Pain** (253) lead the list. These are common, often non-specific adverse events that may reflect both true drug effects and reporting bias. COVID-19 (140) appears in the top 20, consistent with pandemic-era FAERS data.

In [35]:
df = run_query("""
MATCH (c:Case)-[:HAS_REACTION]->(r:Reaction)
WHERE r.description IS NOT NULL
RETURN r.description AS reaction, count(*) AS frequency
ORDER BY frequency DESC
LIMIT 20;
""")
display(df)

,reaction,frequency
0,Fatigue,303
1,Product dose omission issue,285
2,Headache,272
3,Nausea,256
4,Pain,253
5,Dyspnoea,245
6,Pneumonia,229
7,Diarrhoea,219
8,Fall,198
9,Off label use,196


### 2) Top 10 drugs associated with severe outcomes

**Query logic:** The graph models suspect roles as **Case pointing to Drug** via `IS_PRIMARY_SUSPECT` or `IS_SECONDARY_SUSPECT` relationships (e.g. `(c:Case)-[:IS_PRIMARY_SUSPECT|IS_SECONDARY_SUSPECT]->(d:Drug)`), and case outcome as `(c:Case)-[:RESULTED_IN]->(o:Outcome)` with the outcome label in `o.outcome` (EDA 8). I match drugs that are linked to cases as either primary or secondary suspects, where those cases have one of the four severe outcomes: `Death`, `Life-Threatening`, `Disability`, and `Hospitalization - Initial or Prolonged` (exact strings as in the database). I count **distinct cases** per drug so each patient is counted once even if multiple suspect relationships exist, then order by that count descending and limit to 10. Drug name is returned from `d.name`.

**Explanation:** This ranks drugs by how often they appear in cases with the most serious outcomes. It supports signal detection for regulatory review, labeling, and risk management. The count reflects reported associations, not proven causation; follow-up analysis would account for confounders and exposure.

**Interpretation of results:** **REVLIMID** (218 severe cases) leads by a wide margin, consistent with its use in high-risk oncology (multiple myeloma treatment). **NIVOLUMAB** (82 cases) and **ATEZOLIZUMAB** (77 cases) are checkpoint inhibitor immunotherapies used in oncology, which are known to have serious immune-related adverse events. **HUMAN NORMAL IMMUNOGLOBULIN; LIQUID** (66 cases) and **CUVITRU** (61 cases) are immunoglobulin replacement therapies, likely used in immunocompromised patients who are at higher risk for severe outcomes. **POMALYST** (65 cases) is another oncology drug (multiple myeloma), similar to REVLIMID. **DEXAMETHASONE** (65 cases) is a corticosteroid used in serious conditions including oncology and inflammatory diseases. **CYCLOPHOSPHAMIDE** (64 cases) is a chemotherapy agent. **REMODULIN** (57 cases) is used for pulmonary arterial hypertension, a serious cardiovascular condition. **Teduglutide** (53 cases) is used for short bowel syndrome. The top 10 is dominated by oncology and immunology drugs, reflecting the high-risk patient populations these medications treat. These associations warrant further investigation, not causal inference.

In [49]:
df = run_query("""
MATCH (d:Drug)<-[:IS_PRIMARY_SUSPECT|IS_SECONDARY_SUSPECT]-(c:Case)
MATCH (c)-[:RESULTED_IN]->(o:Outcome)
WHERE o.outcome IN ['Death', 'Life-Threatening', 'Disability', 'Hospitalization - Initial or Prolonged']
RETURN d.name AS drug_name, count(DISTINCT c) AS severe_cases
ORDER BY severe_cases DESC
LIMIT 10;

""")
display(df)

,drug_name,severe_cases
0,REVLIMID,218
1,NIVOLUMAB,82
2,ATEZOLIZUMAB,77
3,HUMAN NORMAL IMMUNOGLOBULIN; LIQUID,66
4,POMALYST,65
5,DEXAMETHASONE,65
6,CYCLOPHOSPHAMIDE,64
7,CUVITRU,61
8,REMODULIN,57
9,Teduglutide,53


### 3) Top 10 manufacturers by number of drugs with reported side effects

**Query logic:** Discovery showed that **Manufacturer** connects to **Case** via `(m:Manufacturer)-[:REGISTERED]->(c:Case)` (not to Drug); the manufacturer identifier is `m.manufacturerName`. I restrict to cases that have at least one reported reaction (`(c)-[:HAS_REACTION]->(r:Reaction)`), then for each such case we collect the drugs linked as any suspect type (`(d:Drug)<-[:IS_PRIMARY_SUSPECT|IS_SECONDARY_SUSPECT|IS_CONCOMITANT|IS_INTERACTING]-(c)`). Per manufacturer we return `m.manufacturerName` and the count of **distinct drugs** across all their registered cases that have a reaction. Order by that count descending and limit to 10.

**Explanation:** This ranks manufacturers by the diversity of drugs (distinct product names) that appear in their registered cases with at least one adverse reaction. It supports comparative oversight—which firms have the broadest set of products with reported side effects in this dataset—and can inform prioritization of inspections or requests for additional safety data. The metric reflects report volume and product mix, not necessarily severity.

**Interpretation of results:** **PFIZER** (617 drugs) and **ROCHE** (595) lead, reflecting large portfolios and high report volume. **CELGENE** (452) and **NOVARTIS** (386) follow. These manufacturers have the widest product diversity in cases with reported reactions, which may warrant prioritization for safety surveillance.

In [37]:
df = run_query("""
MATCH (m:Manufacturer)-[:REGISTERED]->(c:Case)-[:HAS_REACTION]->(r:Reaction)
MATCH (d:Drug)<-[:IS_PRIMARY_SUSPECT|IS_SECONDARY_SUSPECT|IS_CONCOMITANT|IS_INTERACTING]-(c)
RETURN m.manufacturerName AS manufacturer, count(DISTINCT d) AS drugs_with_side_effects
ORDER BY drugs_with_side_effects DESC
LIMIT 10;
""")
display(df)


,manufacturer,drugs_with_side_effects
0,PFIZER,617
1,ROCHE,595
2,CELGENE,452
3,NOVARTIS,386
4,TAKEDA,356
5,ABBVIE,352
6,BRISTOL MYERS SQUIBB,307
7,JOHNSON AND JOHNSON,218
8,GLAXOSMITHKLINE,200
9,AMGEN,191


### 4) Top 5 drugs and their distinct side effects for the leading manufacturer (PFIZER)

**Query logic:** I use the top manufacturer from Question 3 (**PFIZER**, `manufacturerName = 'PFIZER'`). The path is the same as in Q3: `(m:Manufacturer)-[:REGISTERED]->(c:Case)-[:HAS_REACTION]->(r:Reaction)` and `(d:Drug)<-[:IS_PRIMARY_SUSPECT|IS_SECONDARY_SUSPECT|IS_CONCOMITANT|IS_INTERACTING]-(c)`. I filter to non-null `r.description`, then for each drug return `d.name`, `collect(DISTINCT r.description)` as `side_effects`, and `count(DISTINCT c)` as `case_count`. I order by case count descending and limit to 5, so we get the five drugs (in PFIZER’s registered cases with reactions) that appear in the most cases, each with its full list of distinct reported side effects.

**Explanation:** This drills down from Q3’s leading manufacturer to show which of its drugs have the most case volume and what adverse reactions were reported for each. It supports manufacturer-level review and product-specific safety profiles. The `side_effects` list can be long; it is the full set of distinct reaction descriptions for that drug in this subset of cases.

**Interpretation of results:**

- **LYRICA** (120 cases) and **GENOTROPIN** (77) have the highest case counts for PFIZER. 
- LYRICA's side-effect profile includes pain, dizziness, and CNS effects; GENOTROPIN's includes device-related issues and metabolic effects. 
- **IBRANCE**, **XELJANZ XR**, and **XELJANZ** show oncology/immunology reaction patterns (neutropenia, infections, thrombosis). 

The long side-effect lists reflect diverse reported adverse events per drug.

In [59]:
df = run_query("""
MATCH (m:Manufacturer {manufacturerName: 'PFIZER'})-[:REGISTERED]->(c:Case)-[:HAS_REACTION]->(r:Reaction)
MATCH (d:Drug)<-[:IS_PRIMARY_SUSPECT|IS_SECONDARY_SUSPECT|IS_CONCOMITANT|IS_INTERACTING]-(c)
WHERE r.description IS NOT NULL
RETURN d.name AS drug_name,
       collect(DISTINCT r.description) AS side_effects,
       count(DISTINCT c) AS case_count
ORDER BY case_count DESC
LIMIT 5;
""")
display(df)

,drug_name,side_effects,case_count
0,LYRICA,"[Ageusia, Pain, Hypoacusis, Temporomandibular joint syndrome, Depressed level of consciousness, Intentional product use issue, Gingival disorder, Malaise, Anosmia, Viral infection, Deafness, Dry mouth, Confusional state, Memory impairment, Toothache, Panic attack, Anxiety, Hallucination, Dyspnoea, Withdrawal syndrome, Drug ineffective, Feeling abnormal, Feeling of despair, Illness, Condition aggravated, Chest pain, Irritability, Muscle spasms, Headache, Blood pressure increased, Abdominal pain upper, Intentional product misuse, Upper limb fracture, Fall, Onychoclasis, Insomnia, Pollakiuria, Product dispensing error, Nail growth abnormal, Paralysis, Impaired driving ability, Tinnitus, Dysphagia, Sedation, Back pain, Intervertebral disc protrusion, Visual impairment, Pain in extremity, Joint swelling, Arthralgia, Depression, Hypertension, Nasal congestion, Nasopharyngitis, Sinusitis, Product dose omission issue, Nervousness, Reading disorder, Cataract, Eye disorder, Fungal infection, Hot flush, Granuloma annulare, Somnolence, Pain in jaw, Arthritis, Angina pectoris, General physical health deterioration, Chills, Discomfort, Weight decreased, Musculoskeletal stiffness, Body temperature abnormal, Asthenia, Hypersomnia, Influenza like illness, Gait disturbance, Feeling cold, Eating disorder, Heart rate increased, Pyrexia, Nausea, Bone disorder, Movement disorder, Joint stiffness, Thinking abnormal, Migraine, Off label use, Feeling of body temperature change, Dizziness, Sepsis, Osteomyelitis, Prescribed overdose, Gait inability, Death, Sitting disability, Neuralgia, Tremor, Spinal cord compression, Panic reaction, ...]",120
1,GENOTROPIN,"[Device breakage, Drug dose omission by device, Product prescribing error, Device use error, Device issue, Device physical property issue, Product physical issue, Wrong technique in device usage process, Device mechanical issue, Device use issue, Device leakage, Device power source issue, Blindness, Device information output issue, Incorrect dose administered by device, Poor quality device used, Device malfunction, Incorrect dose administered, Product dispensing error, Product dose omission issue, Injection site pain, Expired device used, Product quality issue, Product leakage, Needle issue, Increased appetite, Injection site haemorrhage, Insulin-like growth factor increased, Product storage error, Product colour issue, Poor quality product administered, Device delivery system issue, Off label use, Emotional disorder, Crying, Bone density increased, Device maintenance issue, Intentional device misuse, Bone density decreased, Mood altered, Product dose omission in error, Asthenia, Loss of personal independence in daily activities, Hip fracture, Fatigue, Device difficult to use, Device defective, Suicide attempt, Injection site mass, Weight increased, Headache, Eye disorder, Depression, Product odour abnormal, Disturbance in attention, Withdrawal syndrome, Judgement impaired, Hypoglycaemia, Product prescribing issue, Circumstance or information capable of leading to medication error, Drug level below therapeutic, Tinnitus, Feeling abnormal, Dizziness, Mood swings, Hypotension, Impaired driving ability, Malaise, Confusional state, Dyspnoea, Anxiety, Adrenal insufficiency, Memory impairment, Hepatic steatosis, Accidental overdose, Vitamin D deficiency, Throat tightness, Dysphonia, Amnesia, Chest discomfort, Exposure via skin contact, Senile osteoporosis, Oedema, Peripheral swelling, Fear of injection, Psychological trauma, Visual impairment, Condition aggravated, Nausea, Vomiting, Blood thyroid stimulating hormone increased, Globulins decreased, Protein total decreased, Aspartate aminotransferase increased, Cardiac failure, Pain in extremity, Gait disturbance, Nipple disorder, Device failure]",77
2,IBRANCE,"[Product dose omission issue, Product prescribing error, Neoplasm progression, Neutropenia, Anaemia, Off label use, Thrombocytopenia, White blood cell cou

---
## GDS: Graph Data Science Analysis

The following analyses use Neo4j's Graph Data Science (GDS) library to identify similar patient journeys and patient clusters based on drug and reaction patterns discovered in EDA.

### GDS 1) Node Similarity — Similar Patient Journeys

**Objective:** Identify similar patient journeys where patients experienced identical or highly overlapping sequences of reactions after taking the same drugs.

**Query logic:** I create an in-memory graph projection with Case, Drug, and Reaction nodes, and relationships: Case–Drug (IS_PRIMARY_SUSPECT, IS_SECONDARY_SUSPECT, IS_CONCOMITANT) and Case–Reaction (HAS_REACTION), all undirected. A Case's neighbors are thus its drugs and reactions. I run Node Similarity with Jaccard metric; Jaccard = 1 when two patients have identical neighbor sets (same drugs + same reactions). I filter to Case-to-Case pairs only, then enrich with shared reactions and shared drugs via MATCH. Case identifiers use `primaryid`.

**Explanation:** Jaccard similarity captures both drug overlap and reaction overlap. Pairs with score 1.0 have identical drug–reaction profiles; lower scores indicate partial overlap. This supports finding patients with similar adverse-event journeys for signal detection and cohort analysis.

**Interpretation of results:** 

Pairs with similarity 1.0 share identical drug and reaction sets. Many top pairs involve **REVLIMID** with reactions such as **Pneumonia** or **Neutropenia**, consistent with oncology use and known adverse-event profiles. Pairs with multiple shared drugs (e.g., REVLIMID + DEXAMETHASONE) and shared reactions (e.g., Atrial fibrillation) represent cohorts with similar treatment and adverse-event journeys, useful for signal detection and cohort studies.

In [39]:
df=run_query(
"""
CALL gds.graph.project(
  'patient-journey',
  ['Case','Drug','Reaction'],
  {
    IS_PRIMARY_SUSPECT: { orientation: 'UNDIRECTED' },
    IS_SECONDARY_SUSPECT: { orientation: 'UNDIRECTED' },
    IS_CONCOMITANT: { orientation: 'UNDIRECTED' },
    HAS_REACTION: { orientation: 'UNDIRECTED' }
  }
);
"""
)
display(df)

,nodeProjection,relationshipProjection,graphName,nodeCount,relationshipCount,projectMillis
0,"{'Drug': {'properties': {}, 'label': 'Drug'}, ...",{'IS_CONCOMITANT': {'orientation': 'UNDIRECTED...,patient-journey,9508,73816,908


In [40]:
df = run_query(
"""
CALL gds.nodeSimilarity.stream('patient-journey', {
  similarityMetric: 'JACCARD',
  similarityCutoff: 0.2
})
YIELD node1, node2, similarity
WITH gds.util.asNode(node1) AS c1, gds.util.asNode(node2) AS c2, similarity
WHERE c1:Case AND c2:Case AND elementId(c1) < elementId(c2)
WITH c1, c2, similarity
ORDER BY similarity DESC
LIMIT 10
MATCH (c1)-[:HAS_REACTION]->(r:Reaction)<-[:HAS_REACTION]-(c2)
WITH c1, c2, similarity, collect(DISTINCT r.description) AS shared_reactions
OPTIONAL MATCH (c1)-[:IS_PRIMARY_SUSPECT|IS_SECONDARY_SUSPECT|IS_CONCOMITANT]->(d:Drug)<-[:IS_PRIMARY_SUSPECT|IS_SECONDARY_SUSPECT|IS_CONCOMITANT]-(c2)
WITH c1, c2, similarity, shared_reactions, collect(DISTINCT d.name) AS shared_drugs
RETURN 
  c1.primaryid AS case1,
  c2.primaryid AS case2,
  similarity,
  size(shared_reactions) AS num_shared_reactions,
  shared_reactions[0..5] AS sample_reactions,
  shared_drugs AS shared_drugs
ORDER BY similarity DESC;
""")
display(df)

,case1,case2,similarity,num_shared_reactions,sample_reactions,shared_drugs
0,111530912,158574932,1.0,1,[Pneumonia],[REVLIMID]
1,111530912,124902822,1.0,1,[Pneumonia],[REVLIMID]
2,111140142,147242912,1.0,1,[Neutropenia],[REVLIMID]
3,111140142,194926201,1.0,1,[Neutropenia],[REVLIMID]
4,111530912,164981372,1.0,1,[Pneumonia],[REVLIMID]
5,111530912,124977452,1.0,1,[Pneumonia],[REVLIMID]
6,109837324,125979602,1.0,1,[Atrial fibrillation],"[REVLIMID, DEXAMETHASONE]"
7,111140142,147065782,1.0,1,[Neutropenia],[REVLIMID]
8,111140142,194955072,1.0,1,[Neutropenia],[REVLIMID]
9,112550422,126241282,1.0,1,[Septic shock],"[Carfilzomib, DEXAMETHASONE, REVLIMID]"


### GDS 2) Community detection — patient sub-phenotypes

**Assignment question:** Use a community detection algorithm to cluster patients based on shared demographic characteristics and shared adverse reactions, with the objective of identifying sub-phenotypes of patients who may be at higher risk for specific drug–reaction combinations—i.e., do patients sharing certain demographic characteristics appear to be more likely to suffer from certain adverse reactions?

**In-memory design (no database writes).**

I created the demographic structure—**Gender** virtual nodes and actual **AgeGroup** and **Outcome** nodes—only within a GDS in-memory graph projection. The stored Neo4j database was not modified: no new nodes or relationships were written. **Gender** nodes are virtual (since Gender is a property on Case nodes, not a separate node type), while **AgeGroup** and **Outcome** nodes are actual nodes from the database connected via `FALLS_UNDER` and `RESULTED_IN` relationships respectively. The projection also included **Reaction** and **Drug** so that community structure reflected both demographics and adverse events.

**Weights.** Relationship weights were applied only in the projection: primary suspect (Case–Drug) = 2, secondary suspect = 1.5, and concomitant/interacting = 1. This emphasized primary-suspect drug involvement when clustering. Demographic (Gender, AgeGroup), Case–Reaction, and Case–Outcome edges used weight 1 (or default).

**Algorithm choice: Leiden.**

I used the **Leiden** algorithm because:

- It directly supported the goal of finding patient clusters by shared demographics and reactions, by grouping nodes (Cases, demographic nodes, reactions, drugs) that were densely connected in the projected graph.
- It was well-suited to weighted, multi-type graphs and typically yielded better connected, more interpretable communities than Louvain.
- It required no prior labels or seed communities, which fit the exploratory aim of discovering sub-phenotypes.

I used the following steps: I created the in-memory projection, ran Leiden in stream mode (no write-back), and then described each community by demographics and reactions to answer the assignment question.

In [4]:
run_query("CALL gds.graph.drop('patient-communities', false) YIELD graphName RETURN graphName")

""


In [ ]:
df = run_query("""
CALL () {
  MATCH (c:Case)-[r]->(d:Drug)
  WHERE type(r) IN ['IS_PRIMARY_SUSPECT','IS_SECONDARY_SUSPECT','IS_CONCOMITANT','IS_INTERACTING']
  WITH c, d, r ORDER BY elementId(c), elementId(d), type(r)
  RETURN c AS source, d AS target, labels(c) AS sl, labels(d) AS tl, type(r) AS rt,
    CASE type(r) WHEN 'IS_PRIMARY_SUSPECT' THEN 2.0 WHEN 'IS_SECONDARY_SUSPECT' THEN 1.5 ELSE 1.0 END AS w
  UNION ALL
  MATCH (c:Case)-[r:HAS_REACTION]->(rxn:Reaction)
  WITH c, rxn ORDER BY elementId(c), elementId(rxn)
  RETURN c AS source, rxn AS target, labels(c) AS sl, labels(rxn) AS tl, 'HAS_REACTION' AS rt, 1.0 AS w
  UNION ALL
  MATCH (c:Case) WHERE c.gender IS NOT NULL
  WITH DISTINCT c.gender AS g ORDER BY g
  WITH collect(g) AS genders
  UNWIND range(0, size(genders)-1) AS i
  WITH i, genders[i] AS g, 1000000 + i AS vid
  MATCH (c:Case) WHERE c.gender = g
  WITH c, vid ORDER BY elementId(c)
  RETURN c AS source, vid AS target, labels(c) AS sl, ['Gender'] AS tl, 'HAS_GENDER' AS rt, 1.0 AS w
  UNION ALL
  MATCH (c:Case)-[r:FALLS_UNDER]->(ag:AgeGroup)
  WITH c, ag ORDER BY elementId(c), elementId(ag)
  RETURN c AS source, ag AS target, labels(c) AS sl, labels(ag) AS tl, 'FALLS_UNDER' AS rt, 1.0 AS w
  UNION ALL
  MATCH (c:Case)-[r:RESULTED_IN]->(o:Outcome)
  WITH c, o ORDER BY elementId(c), elementId(o)
  RETURN c AS source, o AS target, labels(c) AS sl, labels(o) AS tl, 'RESULTED_IN' AS rt, 1.0 AS w
}
WITH gds.graph.project(
  'patient-communities',
  source,
  target,
  {
    sourceNodeLabels: sl,
    targetNodeLabels: tl,
    relationshipType: rt,
    relationshipProperties: { weight: w }
  },
  { undirectedRelationshipTypes: ['*'], consecutiveIds: true }
) AS g
RETURN g.graphName AS graphName, g.nodeCount AS nodeCount, g.relationshipCount AS relationshipCount, g.projectMillis AS projectMillis
""")
display(df)

,graphName,nodeCount,relationshipCount,projectMillis
0,patient-communities,9522,99968,1644


### Describe sub-phenotypes by community

I map the streamed Leiden results back to **Case** nodes, then for each community compute: **case count**, **top genders**, **top age groups**, **top outcomes**, and **top adverse reactions**. This answers whether certain demographics associate with specific adverse reactions and outcomes across sub-phenotypes.

In [ ]:
# Map Leiden results to Case nodes and describe each community.

df_communities = run_query("""
CALL gds.leiden.stream(
  'patient-communities',
  { relationshipWeightProperty: 'weight', randomSeed: 9 ,concurrency: 1}
)
YIELD nodeId, communityId
WITH gds.util.asNode(nodeId) AS n, communityId
WHERE n:Case
WITH communityId, collect(n) AS cases
WITH communityId, size(cases) AS case_count, cases

// Store node IDs for re-collecting cases after aggregations
WITH communityId, case_count, cases, [c IN cases | elementId(c)] AS case_ids

// Top Reactions
UNWIND cases AS c
MATCH (c)-[:HAS_REACTION]->(r:Reaction)
WHERE r.description IS NOT NULL
WITH communityId, case_count, case_ids, r.description AS reaction
WITH communityId, case_count, case_ids, reaction, count(*) AS rcnt
ORDER BY communityId, rcnt DESC
WITH communityId, case_count, case_ids, collect(reaction)[0..5] AS top_reactions

// Re-collect cases using stored node IDs
WITH communityId, case_count, top_reactions, case_ids
UNWIND case_ids AS cid
MATCH (c) WHERE elementId(c) = cid
WITH communityId, case_count, top_reactions, case_ids, collect(c) AS cases

// Top Genders
UNWIND cases AS c
WITH communityId, case_count, top_reactions, case_ids, c
WHERE c.gender IS NOT NULL
WITH communityId, case_count, top_reactions, case_ids, c.gender AS gender
WITH communityId, case_count, top_reactions, case_ids, gender, count(*) AS gcnt
ORDER BY communityId, gcnt DESC
WITH communityId, case_count, top_reactions, case_ids, collect(gender)[0..3] AS top_genders

// Re-collect cases using stored node IDs
WITH communityId, case_count, top_reactions, top_genders, case_ids
UNWIND case_ids AS cid
MATCH (c) WHERE elementId(c) = cid
WITH communityId, case_count, top_reactions, top_genders, case_ids, collect(c) AS cases

// Top Age Groups
UNWIND cases AS c
MATCH (c)-[:FALLS_UNDER]->(ag:AgeGroup)
WITH communityId, case_count, top_reactions, top_genders, case_ids, ag.ageGroup AS age_group
WITH communityId, case_count, top_reactions, top_genders, case_ids, age_group, count(*) AS agcnt
ORDER BY communityId, agcnt DESC
WITH communityId, case_count, top_reactions, top_genders, case_ids, collect(age_group)[0..3] AS top_age_groups

// Re-collect cases using stored node IDs
WITH communityId, case_count, top_reactions, top_genders, top_age_groups, case_ids
UNWIND case_ids AS cid
MATCH (c) WHERE elementId(c) = cid
WITH communityId, case_count, top_reactions, top_genders, top_age_groups, case_ids, collect(c) AS cases


RETURN communityId,
       case_count,
       top_genders,
       top_age_groups,
       top_reactions
ORDER BY case_count DESC;

""")
df_communities = df_communities.sort_values('case_count', ascending=False).reset_index(drop=True)
display(df_communities)

,communityId,case_count,top_genders,top_age_groups,top_reactions
0,0,680,"[M, F]","[Elderly, Adult, Adolescent]","[Pneumonia, Diarrhoea, Fatigue, Dyspnoea, Rash]"
1,9,576,"[F, M]","[Adult, Elderly, Adolescent]","[Sinusitis, Headache, Fatigue, Product dose omission issue, Pneumonia]"
2,4,407,"[F, M, U]","[Adult, Elderly, Adolescent]","[Pain, Fall, Drug ineffective, Dizziness, Feeling abnormal]"
3,10,404,"[F, M]","[Adult, Elderly, Adolescent]","[Alopecia, Prostate cancer, Pain, Breast cancer, Psychological trauma]"
4,20,374,"[M, F, U]","[Elderly, Adult, Adolescent]","[Febrile neutropenia, Sepsis, Pneumonia, Pyrexia, Cytokine release syndrome]"
5,22,353,"[F, M, U]","[Adult, Elderly, Adolescent]","[Product dose omission issue, Drug dose omission by device, Device issue, Off label use, Wrong technique in product usage process]"
6,2,266,"[F, M]","[Adult, Elderly, Adolescent]","[No adverse event, Product quality issue, Complication of device insertion, Complication of device removal, Device breakage]"
7,7,225,"[F, M]","[Elderly, Adult, Child]","[Death, Dyspnoea, Hypotension, Pyrexia, Hypoxia]"
8,17,210,"[F, M]","[Elderly, Adult, Child]","[Nausea, Fatigue, Diarrhoea, Vomiting, Dizziness]"
9,13,204,"[F, M, U]","[Elderly, Adult, Adolescent]","[Chronic kidney disease, Acute kidney injury, Renal failure, Gastrooesophageal reflux disease, End stage renal disease]"


**Commentary:** 

The table above summarizes each Leiden community by: 
- number of cases, 
- most frequent genders, 
- most frequent age groups , and 
- most frequent adverse reactions. 

Rows are ordered by *case_count* descending so the largest communities appear first. Communities with a strong over-representation of one gender or age group and a small set of dominant reactions can be interpreted as sub-phenotypes (e.g., a cluster of predominantly elderly cases with similar reaction profiles). Comparing *top_genders*, *top_age_groups*, and *top_reactions* across communities helps answer whether patients sharing certain demographic characteristics appear more likely to suffer from specific adverse reactions; further analysis can drill into drug–reaction combinations for high-risk communities. 



 **Interpretation of results:** 

The Leiden algorithm identified 20 distinct patient sub-phenotypes based on shared demographics and adverse reactions:

- **Community 0** (680 cases) is the largest community: predominantly **Elderly, Adult and Adolescent** patients with mixed gender (**M, F**) showing reactions **Pneumonia, Diarrhoea, Fatigue, Dyspnoea, Rash**—consistent with respiratory and gastrointestinal adverse events in older adults, potentially related to immunosuppressive or oncology treatments.

- **Community 9** (576 cases) shows **Adult,Elderly and Adolescent** patients (**F, M**) with **Sinusitis, Headache, Fatigue, Product dose omission issue, Pneumonia**—likely a respiratory/infection cluster with medication adherence issues, possibly related to chronic conditions requiring regular medication.

- **Community 4** (407 cases) includes **Adult,Elderly and Adolescent** patients across all genders (**F, M, U**) with **Pain, Fall, Drug ineffective, Dizziness, Feeling abnormal**—suggests a general adverse event cluster with non-specific symptoms, possibly related to polypharmacy or medication interactions.

- **Community 10** (404 cases) is **Adult,Elderly and Adolescent** (**F, M**) with **Alopecia, Prostate cancer, Pain, Breast cancer, Psychological trauma**—clearly an oncology sub-phenotype with cancer-specific reactions and treatment-related side effects (alopecia from chemotherapy).

- **Community 20** (374 cases) shows **Elderly, Adult and Adolescent** patients (**M, F, U**) with **Febrile neutropenia, Sepsis, Pneumonia, Pyrexia, Cytokine release syndrome**—a severe infection/immunocompromised sub-phenotype, likely related to oncology or immunosuppressive therapy where patients are at high risk for serious infections.

- **Community 22** (353 cases) includes **Adult,Elderly and Adolescent** (**F, M, U**) with device and medication adherence issues: **Product dose omission issue, Drug dose omission by device, Device issue, Off label use, Wrong technique in product usage process**—a device/medication management sub-phenotype, possibly related to complex medication delivery systems or patient education needs.

- **Community 2** (266 cases) shows **Adult,Elderly and Adolescent** (**F, M**) with **No adverse event, Product quality issue, Complication of device insertion, Complication of device removal, Device breakage**—a device safety reporting cluster, focusing on device-related complications rather than drug reactions.

- **Community 7** (225 cases) includes **Elderly, Adult, and Child** patients (**F, M**) with severe reactions: **Death, Dyspnoea, Hypotension, Pyrexia, Hypoxia**—a critical care sub-phenotype with life-threatening adverse events, warranting urgent safety surveillance.

- **Community 17** (210 cases) shows **Elderly, Adult, and Child** (**F, M**) with gastrointestinal symptoms: **Nausea, Fatigue, Diarrhoea, Vomiting, Dizziness**—a common adverse event cluster, likely related to medications with well-known GI side effects.

- **Community 13** (204 cases) includes **Elderly, Adult, and Adolescent** patients (**F, M, U**) with renal complications: **Chronic kidney disease, Acute kidney injury, Renal failure, Gastrooesophageal reflux disease, End stage renal disease**—a renal sub-phenotype, potentially related to nephrotoxic medications or pre-existing renal conditions.

- **Smaller communities** (24, 5, 8, 25, 6, 12, 28, 3, 21, 30) range from 32–130 cases and show specialized patterns:
  - **Community 5** (92 cases): **COVID-19 pneumonia, Headache, Fatigue, Multiple sclerosis, COVID-19**—pandemic-era cluster with COVID-related adverse events.
  - **Community 6** (57 cases): **Alanine aminotransferase increased, Aspartate aminotransferase increased**—hepatotoxicity sub-phenotype.
  - **Community 12** (56 cases): **Hereditary angioedema**—rare disease cluster.
  - **Community 28** (38 cases): **Embolism, Blindness, Uveitis**—vascular and ocular complications.
  - **Community 3** (37 cases): **Febrile neutropenia, Pyrexia, Vomiting, Dehydration, Hypoglycaemia**—pediatric/adolescent oncology cluster.

**Key findings:**
- **Age-group patterns** more clearly differentiate risk profiles than gender (which is generally mixed across communities).
- **Geriatric sub-phenotypes** (Elderly-dominated communities) show higher rates of respiratory infections (Pneumonia), sepsis, and severe outcomes.
- **Oncology-related communities** (Communities 10, 20) show characteristic reactions like alopecia, febrile neutropenia, and cytokine release syndrome.
- **Device-related communities** (Communities 22, 2) highlight medication delivery and device safety issues.
- The algorithm successfully identified clinically meaningful sub-phenotypes that could inform targeted safety surveillance and patient risk stratification.

## 7. Conclusion

This approach leverages an iterative discovery methodology, starting with foundational EDA to understand the graph structure, progressing through analytical queries that provide healthcare insights, and culminating in advanced GDS analyses that identify patient risk patterns. By anticipating challenges around schema discovery, query performance, and algorithm selection, I can proactively address potential issues and ensure a thorough, well-documented analysis.

### Use of AI tools and LLMs

For this assignment, I used large language models (LLMs) as a reference to compare available similarity and community detection algorithms in Neo4j’s GDS library. I used their summaries of pros and cons to help select Jaccard‑based Node Similarity and the Leiden community detection algorithm, and then validated these choices against the FAERS use case and official Neo4j documentation.

The transcript to ChatGPT is https://chatgpt.com/share/69a4a840-46b4-8007-958c-f2b19b50679d and the transcript from Cursor is below.

---

**User**

can you compare the available node similarity algorithms in neo4j

---

**Cursor**

### High-level comparison

Neo4j’s Graph Data Science (GDS) library doesn’t have a huge list of totally separate “node similarity algorithms”; instead it has:

- a **Node Similarity** algorithm (neighbors-based)
- a **K-Nearest Neighbors (KNN)** algorithm (property / vector-based)
- several **similarity functions** (Jaccard, Overlap, Cosine, Pearson, etc.) that these algorithms can use under the hood

Below is a concise comparison focused on how you’d choose between them.

---

### 1. Node Similarity (neighbors-based)

- **What it does**: Compares nodes based on **their neighbors** (e.g. two users are similar if they interact with many of the same items / people).
- **Typical use-case**: “Users who viewed similar products”, “authors with similar co-authors”, “papers citing similar papers”.
- **How it works**:  
  - Builds neighbor sets (optionally weighted) from relationships.  
  - Computes pairwise similarity, typically sparsified (top-k or threshold).
- **Available similarity metrics**:
  - **Jaccard** (set-based): intersection / union of neighbor sets.
  - **Overlap coefficient**: intersection / min(|A|, |B|); favors nodes with smaller degree but strong shared neighbors.
  - **Cosine similarity**: treats neighbor weights as vectors; best when relationships have **weights** (e.g. rating counts, interaction frequency).
- **When to use**:
  - Your notion of similarity is “share many neighbors”.
  - You already have a bipartite or general interaction graph.
  - You want relationships like `(:User)-[:SIMILAR_TO]->(:User)` based on shared items.

---

### 2. K-Nearest Neighbors (KNN) (property/vector-based)

- **What it does**: Compares nodes by **numeric properties or embedding vectors** (e.g. feature vectors, FastRP / Node2Vec / GraphSAGE embeddings).
- **Typical use-case**: “Find similar products by embedding vector”, “similar documents by text embedding”, “similar users by feature vector”.
- **How it works**:
  - Reads one or more numeric properties (often an embedding array).
  - Uses a distance/similarity function to find each node’s **k nearest neighbors**.
  - Creates relationships like `(:Item)-[:SIMILAR_TO {score}]->(:Item)`.
- **Available similarity/distance measures** (via similarity functions):
  - **Cosine similarity** (most common for embeddings).
  - **Euclidean distance / Euclidean similarity**.
  - **Pearson** (for correlation-style similarity on numeric vectors).
- **When to use**:
  - You already have or plan to compute **embeddings** (FastRP, Node2Vec, GraphSAGE, external ML).
  - Similarity should reflect **latent structure** or rich features, not just shared neighbors.
  - You need a fixed **k** similar nodes per node, e.g. for recommendation or ANN-style queries.

---

### 3. Standalone similarity functions (arrays / properties)

These are lower-level **functions** that can be used directly or inside custom pipelines:

- **Categorical / set-based**:
  - **Jaccard**: for overlap between sets (IDs, categories, tags).
  - **Overlap coefficient**: emphasizes strong overlap when one set is much smaller.
- **Numeric / vector-based**:
  - **Cosine**: angle between vectors; ideal for high-dimensional embeddings.
  - **Pearson**: correlation; good when you care about shape/trend rather than absolute values.
  - **Euclidean distance / similarity**: direct geometric distance.

They’re the building blocks behind `gds.nodeSimilarity` and `gds.knn`.

---

### Choosing between them

- **You have a graph of interactions / co-occurrences** (e.g. users–items, authors–papers):
  - **Use `gds.nodeSimilarity`** with:
    - **Jaccard / Overlap** for unweighted edges and pure “co-occurrence” style similarity.
    - **Cosine** if relationships are **weighted**.

- **You have feature vectors or embeddings on nodes**:
  - **Use `gds.knn`** on a vector property, with:
    - **Cosine** for embeddings (most standard).
    - **Euclidean** when your feature space is naturally Euclidean.

- **You’re building a custom scoring logic in Cypher or a GDS pipeline**:
  - Use **similarity functions** (Jaccard, Cosine, Pearson, etc.) directly on arrays or vectors.

---

If you tell me your specific use-case (e.g. “I have students and courses and I want similar students” or “I have article embeddings”), I can recommend a precise algorithm + config and sketch the Cypher / GDS calls for it.

---

**User**

great work, lets also compare community detection algorithms

---

**Cursor**

### Overview

Neo4j GDS has several core **community detection** algorithms, each with different trade-offs in speed, quality, and assumptions about communities.

---

### 1. Leiden

- **Goal**: High-quality, well-connected communities via **modularity optimization**, improving on Louvain.
- **Strengths**:
  - **Better community quality** and fewer “broken” communities than Louvain.
  - Handles large graphs; hierarchical (can give communities at multiple resolutions).
- **Weaknesses**:
  - More complex than Label Propagation; a bit heavier than LPA.
- **Use when**:  
  - You care about **community quality** and **well-connectedness**.
  - You’re okay with a bit more compute for better structure.

---

### 2. Louvain

- **Goal**: Detect communities by **maximizing modularity**.
- **Strengths**:
  - Well-known, widely used; good quality vs performance.
  - Hierarchical: gives multiple levels of communities.
- **Weaknesses**:
  - Can produce communities that are **not well connected internally** (one reason Leiden was introduced).
- **Use when**:
  - You want modularity-based clustering and don’t need Leiden’s extra guarantees.
  - You want a good default and performance is more important than the last bit of quality.

---

### 3. Label Propagation (LPA)

- **Goal**: Very fast, approximate communities by **propagating labels** among neighbors.
- **Strengths**:
  - **Very fast and scalable**, simple to run.
  - No need to set an objective like modularity.
- **Weaknesses**:
  - More **unstable** (results can vary run-to-run).
  - Community quality generally lower than Louvain/Leiden; can give very coarse or noisy partitions.
- **Use when**:
  - You need a **quick, cheap** community detection.
  - You want a first-pass structure, then maybe refine with another method.

---

### 4. Connected Components (WCC / SCC)

- **Goal**: Find **connected subgraphs**, not “communities” in the modularity sense.
  - **WCC**: Weakly Connected Components (ignores edge direction).
  - **SCC**: Strongly Connected Components (direction-respecting, each node reachable from each other).
- **Strengths**:
  - Extremely **fast and simple**.
  - Gives a hard partition based solely on connectivity.
- **Weaknesses**:
  - Doesn’t consider density or modularity; large components may still contain many sub-communities.
- **Use when**:
  - You need the **basic connectivity structure**: which nodes are in the same “island”.
  - As a **preprocessing** step (e.g., run Louvain/Leiden/LPA inside each WCC).

---

### How to choose (rule of thumb)

- **Best overall quality**: **Leiden**  
- **Balanced & classic**: **Louvain**
- **Fastest approximate grouping**: **Label Propagation**
- **Pure connectivity partitioning**: **Connected Components (WCC/SCC)**

If you describe your graph (directed/undirected, size, and what “community” means in your assignment), I can suggest a specific algorithm and show the exact GDS calls you’d use.

